### Imports

In [9]:
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv
from testgen.utils import *

In [10]:
base_path = Path().cwd()
load_dotenv(override=True)

True

In [11]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

# Find Examples

In [12]:
examples_txt, examples = get_example_txt(df_examples, 1)

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]

Requirement: The power steering system must adapt the torque levels in response to detected road surface conditions (e.g., ice, water, gravel)
Target Sensor/s: [steering_torque]

Requirement: The control system must detect and mitigate power oversteer to maintain vehicle control by adjusting the wheel steering angle and acceleration pedal inputs
Target Sensor/s: [acceleration_pedal, wheel_steering_angle]




# Response Format

In [13]:
from testgen.prompts.Sensors import Sensors
from pydantic import Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


VectorFormat = create_model("VectorFormat", **sensor_attrs)

# LLM

In [14]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPrompt

In [19]:
llm_models = {
    # "azure": ["gpt-4o-mini"],
    # "azure": ["gpt-4o-mini", "gpt-4o"],
    "novita": [
        # "qwen/qwen2.5-7b-instruct",
        "google/gemma-3-27b-it",
        "meta-llama/llama-3-70b-instruct",
    ],
}

endpoint_attrs = {
    "azure": {
        "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "api_version": os.getenv("AZURE_API_VERSION"),
        "base_url": os.getenv("AZURE_OPENAI_ENDPOINT"),
    },
    "novita": {
        "api_key": os.getenv("NOVITA_API_KEY"),
        "base_url": os.getenv("NOVITA_ENDPOINT"),
    },
}

# Run for all Requirements

In [20]:
for n_example in [3, 5, 8]:

    examples_txt, examples = get_example_txt(df_examples, n_example)

    for endpoint_name in llm_models.keys():

        for model_name in llm_models[endpoint_name]:

            print(f"Running {model_name} on {endpoint_name} examples {n_example}...")

            client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

            results = client_invoke_sensor(
                endpoint_name,
                model_name,
                client,
                df,
                SystemPrompt,
                Sensors,
                examples_txt,
                UserPrompt,
                response_format=VectorFormat,
            )

            (
                number_of_requests,
                accuracy,
                avg_time_per_req,
                avg_token_per_req,
                avg_completion_token_per_req,
                total_tokens,
                total_completion_tokens,
                total_time,
            ) = calc_stats(results)

            results_file = save_responses(
                base_path,
                "sensor_single",
                model_name=model_name.split("/")[-1],
                n_examples=n_example,
                examples=examples,
                accuracy=accuracy,
                number_of_requests=number_of_requests,
                total_tokens=total_tokens,
                total_completion_tokens=total_completion_tokens,
                avg_token_per_req=avg_token_per_req,
                avg_completion_token_per_req=avg_completion_token_per_req,
                avg_time_per_req=avg_time_per_req,
                results=results,
            )

            print(f"Done")

Running google/gemma-3-27b-it on novita examples 3...


  0%|          | 0/97 [00:00<?, ?it/s]

JSONDecodeError: Expecting ',' delimiter: line 2042 column 1 (char 6162)